# UFO datetime fix

# IMPORTS

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import os, sys

import matplotlib.pyplot as plt
import matplotlib
import numpy as np

import re


In [2]:
import ipywidgets
from tqdm import tqdm

# PARAMETERS

In [3]:

OUT_DIR = Path('/home/cefect/LS/10_IO/UFO')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# LOAD DATA

## STAC files

downloaded from https://zenodo.org/records/15238470

In [4]:
search_fp = Path('/home/cefect/LS/10_IO/UFO/inputs/labels')
assert search_fp.exists(), f"Input folder {search_fp} does not exist"

scan the directory and concat all the stac jsons

In [5]:


records = []
for dirpath, dirs, files in tqdm(os.walk(search_fp), desc="Processing directories"):
    for f in files:
        if f.endswith(".json"):
            fp = os.path.join(dirpath, f)
            try:
                gdf = gpd.read_file(fp)
                records.append(gdf)
            except Exception:
                pass

concat_gdf = gpd.pd.concat(records, ignore_index=True)
concat_gdf = concat_gdf.set_crs(4326)
concat_gdf.info()

Processing directories: 232it [00:00, 645.69it/s]

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   id                231 non-null    object             
 1   Event             231 non-null    object             
 2   Chip_ID           231 non-null    object             
 3   Image source      231 non-null    object             
 4   Data Description  231 non-null    object             
 5   license           231 non-null    object             
 6   Contact           231 non-null    object             
 7   Mission           231 non-null    object             
 8   gsd               231 non-null    int32              
 9   eo:bands          231 non-null    object             
 10  datetime          231 non-null    datetime64[ms, UTC]
 11  geometry          231 non-null    geometry           
dtypes: datetime64[ms, UTC](1), geometry(1), int32(1), object

In [6]:
concat_gdf.head()

,id,Event,Chip_ID,Image source,Data Description,license,Contact,Mission,gsd,eo:bands,datetime,geometry
0,MID_b23w,UrbanFloodObservations,MID_b23w,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((-84.26685 43.59507, -84.26685 43.629..."
1,SPS_ogkc,UrbanFloodObservations,SPS_ogkc,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((397236 1705113, 397236 1708185, 4003..."
2,PNE_45gc,UrbanFloodObservations,PNE_45gc,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((-96.0276 41.06194, -96.0276 41.09696..."
3,SLC_vb4f,UrbanFloodObservations,SLC_vb4f,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((-56.38669 -34.42442, -56.38669 -34.3..."
4,CTO_xs1y,UrbanFloodObservations,CTO_xs1y,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((105.83711 9.94959, 105.83711 9.97756..."


light post to get columns for join

In [7]:
concat_gdf = (
    concat_gdf
    .assign(
        UFO_event_id = lambda df: df['id'].str.split('_').str[0],
        ps_id = lambda df: df['id'].str.split('_').str[1],
    )
)

concat_gdf[['UFO_event_id', 'ps_id']]

,UFO_event_id,ps_id
0,MID,b23w
1,SPS,ogkc
2,PNE,45gc
3,SLC,vb4f
4,CTO,xs1y
...,...,...
226,HTX,fjhm
227,DKA,0705
228,HTX,z4st
229,CMO,vk0m


In [8]:
concat_gdf.drop(columns=['geometry'])

,id,Event,Chip_ID,Image source,Data Description,license,Contact,Mission,gsd,eo:bands,datetime,UFO_event_id,ps_id
0,MID_b23w,UrbanFloodObservations,MID_b23w,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,MID,b23w
1,SPS_ogkc,UrbanFloodObservations,SPS_ogkc,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,SPS,ogkc
2,PNE_45gc,UrbanFloodObservations,PNE_45gc,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,PNE,45gc
3,SLC_vb4f,UrbanFloodObservations,SLC_vb4f,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,SLC,vb4f
4,CTO_xs1y,UrbanFloodObservations,CTO_xs1y,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,CTO,xs1y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
226,HTX_fjhm,UrbanFloodObservations,HTX_fjhm,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,HTX,fjhm
227,DKA_0705,UrbanFloodObservations,DKA_0705,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,DKA,0705
228,HTX_z4st,UrbanFloodObservations,HTX_z4st,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,HTX,z4st
229,CMO_vk0m,UrbanFloodObservations,CMO_vk0m,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,CMO,vk0m


## Scene metadata

gsheet from Rohit

In [9]:
fp = Path('./PSnames.csv')
assert fp.exists(), f"File {fp} does not exist"

In [10]:
meta_df_raw = pd.read_csv(fp)
meta_df_raw.head()


,full_filename,short_filename
0,10_20190326_072743_1021_1ynv.tif,1ynv.tif
1,10_20190326_072743_1021_a69z.tif,a69z.tif
2,06_20190321_155335_0f2b_17rl.tif,17rl.tif
3,06_20190321_155335_0f2b_f51l.tif,f51l.tif
4,06_20190321_155335_0f2b_go0s.tif,go0s.tif


In [11]:
#extract info from filename
df = meta_df_raw
meta_df = (
    meta_df_raw
    .assign(
        ps_id=lambda df: df['short_filename'].str.replace('.tif', '', regex=False),
        date=lambda df: pd.to_datetime(
            df['full_filename'].str.split('_').str[1],
            format='%Y%m%d', errors='coerce'),
    )
)
print(meta_df.info())
meta_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   full_filename   378 non-null    object        
 1   short_filename  378 non-null    object        
 2   ps_id           378 non-null    object        
 3   date            378 non-null    datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 11.9+ KB
None


,full_filename,short_filename,ps_id,date
0,10_20190326_072743_1021_1ynv.tif,1ynv.tif,1ynv,2019-03-26
1,10_20190326_072743_1021_a69z.tif,a69z.tif,a69z,2019-03-26
2,06_20190321_155335_0f2b_17rl.tif,17rl.tif,17rl,2019-03-21
3,06_20190321_155335_0f2b_f51l.tif,f51l.tif,f51l,2019-03-21
4,06_20190321_155335_0f2b_go0s.tif,go0s.tif,go0s,2019-03-21


In [12]:
#investigate null dates
bx = meta_df['date'].isnull()
if bx.any():
    meta_df[bx]

# JOIN

In [13]:
meta_set = set(meta_df['ps_id'].unique())
concat_set = set(concat_gdf['ps_id'].unique())
missing_in_meta = concat_set - meta_set
missing_in_concat = meta_set - concat_set
print(f"Number of ps_id in concat_gdf not in meta_df: {len(missing_in_meta)}")
print(f"Number of ps_id in meta_df not in concat_gdf: {len(missing_in_concat)}")

Number of ps_id in concat_gdf not in meta_df: 0
Number of ps_id in meta_df not in concat_gdf: 147


In [14]:

join_df = (
    concat_gdf.set_index('ps_id', verify_integrity=True)
    .join(
        meta_df.set_index('ps_id', verify_integrity=True)['date'].rename('datetime_fixed'),
        how='left',
    )
)
print(join_df.info())
join_df.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 231 entries, b23w to lipf
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   id                231 non-null    object             
 1   Event             231 non-null    object             
 2   Chip_ID           231 non-null    object             
 3   Image source      231 non-null    object             
 4   Data Description  231 non-null    object             
 5   license           231 non-null    object             
 6   Contact           231 non-null    object             
 7   Mission           231 non-null    object             
 8   gsd               231 non-null    int32              
 9   eo:bands          231 non-null    object             
 10  datetime          231 non-null    datetime64[ms, UTC]
 11  geometry          231 non-null    geometry           
 12  UFO_event_id      231 non-null    object             
 13

,id,Event,Chip_ID,Image source,Data Description,license,Contact,Mission,gsd,eo:bands,datetime,geometry,UFO_event_id,datetime_fixed
ps_id,,,,,,,,,,,,,,
b23w,MID_b23w,UrbanFloodObservations,MID_b23w,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((-84.26685 43.59507, -84.26685 43.629...",MID,2020-05-20
ogkc,SPS_ogkc,UrbanFloodObservations,SPS_ogkc,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((397236 1705113, 397236 1708185, 4003...",SPS,2020-11-25
45gc,PNE_45gc,UrbanFloodObservations,PNE_45gc,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((-96.0276 41.06194, -96.0276 41.09696...",PNE,2019-03-18
vb4f,SLC_vb4f,UrbanFloodObservations,SLC_vb4f,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((-56.38669 -34.42442, -56.38669 -34.3...",SLC,2019-06-20
xs1y,CTO_xs1y,UrbanFloodObservations,CTO_xs1y,Hand label,0: None-inundation. 1: Inundation,CC-BY-SA-4.0,rohitmukherjee@live.com,This Urban Flood Observations dataset is a man...,3,"[ { ""name"": ""Value 0"", ""description"": ""None-in...",1970-01-01 00:00:00+00:00,"POLYGON ((105.83711 9.94959, 105.83711 9.97756...",CTO,2019-10-02


## write

In [16]:
ofp = 'UFO_events_analysis_ready.geojson'
gdf = (
    join_df
    .reset_index()
    .loc[:, ['ps_id', 'UFO_event_id', 'Chip_ID', 'datetime_fixed', 'geometry']]
    .rename(columns={'datetime_fixed': 'datetime'})
    .sort_values(['UFO_event_id', 'ps_id'])
)

gpd.GeoDataFrame(gdf, crs='4326').to_file(ofp, driver='GeoJSON')
print(f"Wrote output to {ofp}")

Wrote output to UFO_events_analysis_ready.geojson
